# 06 - Rolling / Walk-Forward Optimization

Re-optimize portfolio weights over rolling time windows and evaluate out-of-sample performance.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.data import fetch_prices, compute_returns
from src.rolling import (
    rolling_optimize,
    walk_forward_test_returns,
    rolling_weights_over_time,
    compare_rolling_strategies,
)
from src.portfolio import build_portfolio_returns, compare_portfolios, equity_curve
from src.metrics import metrics_table, sharpe_ratio, annualized_return

In [ ]:
tickers = ["AAPL", "MSFT", "VWCE.MI", "BTC-USD"]
start_date = "2016-01-01"
end_date = "2025-01-01"

prices = fetch_prices(tickers, start=start_date, end=end_date)
returns = compute_returns(prices)
print(f"Loaded {len(prices)} trading days")

## Configure Rolling Optimization

Choose the optimization target and window sizes.

In [ ]:
target = "max_sharpe"
train_window = 504  # ~2 years of trading days
test_window = 21    # ~1 month
step = 21           # rebalance every month

results = rolling_optimize(
    prices, target=target,
    train_window=train_window, test_window=test_window,
    step=step, risk_free_rate=0.04,
)
print(f"Generated {len(results)} rolling windows")
results[['date', 'train_sharpe', 'train_return', 'train_volatility', 'test_return']].head(10)

## Weights Over Time

In [ ]:
weights_df = rolling_weights_over_time(results)

fig = go.Figure()
for col in weights_df.columns:
    fig.add_trace(go.Scatter(
        x=weights_df.index, y=weights_df[col] * 100,
        name=col, mode='lines', stackgroup='one',
    ))
fig.update_layout(
    title=f'Rolling Weights Over Time (target: {target})',
    yaxis_title='Weight %', xaxis_title='Date',
    template='plotly_white', height=500,
)
fig.show()

## Train vs Test Performance

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Train Sharpe (In-Sample)', 'Test Return (Out-of-Sample)'),
                    vertical_spacing=0.08)

fig.add_trace(go.Bar(
    x=results['date'], y=results['train_sharpe'], name='Train Sharpe',
    marker_color='blue', opacity=0.7,
), row=1, col=1)

fig.add_trace(go.Bar(
    x=results['date'], y=results['test_return'] * 100, name='Test Return %',
    marker_color=['green' if r > 0 else 'red' for r in results['test_return']],
    opacity=0.7,
), row=2, col=1)

fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=1)
fig.update_layout(
    title=f'Rolling Performance (target: {target}, train: {train_window}d, test: {test_window}d)',
    template='plotly_white', height=700,
)
fig.show()

## Walk-Forward Equity Curve

Compare the walk-forward strategy against buy-and-hold benchmarks.

In [ ]:
wf_returns = walk_forward_test_returns(results, prices)

if len(wf_returns) > 0:
    equal_weights = {t: 1.0/len(tickers) for t in tickers}
    equal_ret = build_portfolio_returns(prices, equal_weights)

    aligned_equal = equal_ret.reindex(wf_returns.index).dropna()
    aligned_wf = wf_returns.reindex(aligned_equal.index).dropna()

    curves = pd.DataFrame({
        f'Walk-Forward ({target})': equity_curve(aligned_wf, 10000),
        'Equal Weight': equity_curve(aligned_equal, 10000),
    })

    fig = go.Figure()
    for col in curves.columns:
        fig.add_trace(go.Scatter(x=curves.index, y=curves[col], name=col, mode='lines'))
    fig.update_layout(
        title='Walk-Forward vs Equal Weight',
        yaxis_title='Portfolio Value ($)', xaxis_title='Date',
        template='plotly_white', height=600,
    )
    fig.show()

    comparison = {
        f'Walk-Forward ({target})': aligned_wf,
        'Equal Weight': aligned_equal,
    }
    metrics_table(comparison).style.format('{:.4f}')
else:
    print("Not enough data for walk-forward analysis")

## Compare Multiple Rolling Strategies

In [ ]:
targets_to_compare = ["max_sharpe", "min_volatility", "hrp", "min_cvar"]

all_rolling = compare_rolling_strategies(
    prices, targets=targets_to_compare,
    train_window=train_window, test_window=test_window,
    step=step, risk_free_rate=0.04,
)

summary = {}
for name, res in all_rolling.items():
    wf_ret = walk_forward_test_returns(res, prices)
    if len(wf_ret) > 0:
        summary[name] = {
            'Avg Train Sharpe': res['train_sharpe'].mean(),
            'Avg Test Return': res['test_return'].mean() * 100,
            'Test Win Rate': (res['test_return'] > 0).mean() * 100,
            'WF Total Return': ((1 + wf_ret).prod() - 1) * 100,
            'WF Sharpe': sharpe_ratio(wf_ret),
            'WF Ann. Return': annualized_return(wf_ret),
            'N Periods': len(res),
        }

summary_df = pd.DataFrame(summary)
summary_df.style.format('{:.4f}')

In [ ]:
fig = go.Figure()
metrics_to_plot = ['Avg Train Sharpe', 'Avg Test Return', 'Test Win Rate', 'WF Sharpe']
for metric in metrics_to_plot:
    fig.add_trace(go.Bar(
        x=list(summary.keys()),
        y=[summary[k][metric] for k in summary],
        name=metric,
    ))
fig.update_layout(
    title='Rolling Strategy Comparison',
    barmode='group', template='plotly_white', height=500,
)
fig.show()